# Neighborhoods Exploratory Data Analysis

This notebook performs an exploratory data analysis on Barcelona neighborhoods, focusing on income levels, population, and geographic visualization.

## 1. Import Required Libraries

In [2]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins
import json
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 2. Load Neighborhoods Data

In [1]:
import pandas as pd

# Point this to the file you want to check
FILE_PATH = "../data/neighborhoods/neighborhoods.parquet" 

def inspect_parquet(file_path):
    print(f"📂 Inspecting: {file_path}\n")
    
    try:
        # 1. Load the Parquet file into a Pandas DataFrame
        df = pd.read_parquet(file_path)
        
        # 2. Verify the columns
        print("📊 Columns:")
        print(df.columns.tolist())
        print("-" * 40)
        
        # 3. Check the Data Types (useful to ensure numbers aren't saved as strings)
        print("🔠 Data Types:")
        print(df.dtypes)
        print("-" * 40)
        
        # 4. Check the size of the dataset (rows, columns)
        print(f"📏 Shape: {df.shape[0]} rows, {df.shape[1]} columns")
        print("-" * 40)
        
        # 5. Peek at the first 5 rows to make sure the data looks correct
        print("👀 First 5 rows:")
        print(df.head())

    except Exception as e:
        print(f"❌ Error reading file: {e}")

if __name__ == "__main__":
    inspect_parquet(FILE_PATH)

📂 Inspecting: ../data/neighborhoods/neighborhoods.parquet

❌ Error reading file: Repetition level histogram size mismatch


## 3. Explore Data Structure

In [ ]:
# Display neighborhoods data
print("=== Neighborhoods Geographic Data ===")
print(neighborhoods_gdf.head())
print(f"\nData types:\n{neighborhoods_gdf.dtypes}")
print(f"\nColumns: {neighborhoods_gdf.columns.tolist()}")

print("\n\n=== Neighborhoods Population Data ===")
print(neighborhoods_pop.head())
print(f"\nColumns: {neighborhoods_pop.columns.tolist()}")

print("\n\n=== Income Data ===")
print(income_df.head())
print(f"\nColumns: {income_df.columns.tolist()}")

In [ ]:
# Merge all data for comprehensive analysis
# First, let's prepare the data by merging income and population info
analysis_df = income_df.copy()

# Merge with population data if they have a common key
if len(neighborhoods_pop) > 0:
    # Show the first few rows to understand the structure
    print("Population data structure:")
    print(neighborhoods_pop.head())
    
# Prepare a comprehensive dataset combining all information
print(f"\nTotal neighborhoods in geographic data: {len(neighborhoods_gdf)}")
print(f"Total neighborhoods in income data: {len(income_df)}")

## 4. Income Analysis by Neighborhood

In [ ]:
# Income descriptive statistics
print("=== Income Statistics ===")
print(income_df.describe())

# Identify income columns (numeric columns for analysis)
income_cols = income_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric income columns: {income_cols}")

In [ ]:
# Visualize income distribution across neighborhoods
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Income by neighborhood (bar chart)
if len(income_cols) > 0 and 'neighborhood' in income_df.columns:
    income_sorted = income_df.sort_values(income_cols[0], ascending=False)
    axes[0, 0].barh(income_sorted['neighborhood'].astype(str), income_sorted[income_cols[0]], color='steelblue')
    axes[0, 0].set_xlabel(income_cols[0])
    axes[0, 0].set_title(f'Income Index by Neighborhood')
    axes[0, 0].tick_params(axis='y', labelsize=8)

# Plot 2: Income distribution histogram
if len(income_cols) > 0:
    axes[0, 1].hist(income_df[income_cols[0]], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Income Index Value')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title(f'Distribution of {income_cols[0]}')

# Plot 3: Box plot for income
if len(income_cols) > 0:
    axes[1, 0].boxplot([income_df[col] for col in income_cols], labels=income_cols)
    axes[1, 0].set_ylabel('Value')
    axes[1, 0].set_title('Income Indices Distribution')
    axes[1, 0].tick_params(axis='x', rotation=45)

# Plot 4: Missing values heatmap
axes[1, 1].imshow(income_df.isna().astype(int), cmap='RdYlGn_r', aspect='auto')
axes[1, 1].set_title('Missing Values Heatmap')
axes[1, 1].set_xlabel('Columns')

plt.tight_layout()
plt.show()

print(f"Highest income neighborhood: {income_df[income_cols[0]].idxmax() if len(income_cols) > 0 else 'N/A'}")
print(f"Lowest income neighborhood: {income_df[income_cols[0]].idxmin() if len(income_cols) > 0 else 'N/A'}")

## 5. Population Analysis by Neighborhood

In [ ]:
# Population descriptive statistics
print("=== Population Statistics ===")
print(neighborhoods_pop.describe())

# Identify population columns
pop_cols = neighborhoods_pop.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric population columns: {pop_cols}")
print(f"\nNeighborhoods population data columns: {neighborhoods_pop.columns.tolist()}")


In [ ]:
# Visualize population across neighborhoods
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Population by neighborhood (bar chart)
if len(pop_cols) > 0:
    pop_sorted = neighborhoods_pop.sort_values(pop_cols[0], ascending=False).head(20)
    axes[0, 0].barh(pop_sorted.index.astype(str) if not isinstance(pop_sorted.index, pd.RangeIndex) 
                   else range(len(pop_sorted)), pop_sorted[pop_cols[0]], color='coral')
    axes[0, 0].set_xlabel(pop_cols[0])
    axes[0, 0].set_title(f'Top 20 Neighborhoods by {pop_cols[0]}')
    axes[0, 0].tick_params(axis='y', labelsize=8)

# Plot 2: Population distribution histogram
if len(pop_cols) > 0:
    axes[0, 1].hist(neighborhoods_pop[pop_cols[0]], bins=20, color='coral', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel(f'{pop_cols[0]} Value')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title(f'Distribution of {pop_cols[0]}')

# Plot 3: Box plot for population
if len(pop_cols) > 0:
    axes[1, 0].boxplot([neighborhoods_pop[col] for col in pop_cols[:5]], labels=pop_cols[:5])
    axes[1, 0].set_ylabel('Value')
    axes[1, 0].set_title('Population Indices Distribution')
    axes[1, 0].tick_params(axis='x', rotation=45)

# Plot 4: Cumulative population
if len(pop_cols) > 0:
    pop_sorted_cum = neighborhoods_pop[pop_cols[0]].sort_values(ascending=False)
    cum_sum = pop_sorted_cum.cumsum()
    axes[1, 1].plot(range(len(cum_sum)), cum_sum.values, marker='o', linewidth=2, markersize=6)
    axes[1, 1].set_xlabel('Neighborhoods (sorted by population)')
    axes[1, 1].set_ylabel('Cumulative Population')
    axes[1, 1].set_title('Cumulative Population Distribution')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if len(pop_cols) > 0:
    print(f"\nTotal population: {neighborhoods_pop[pop_cols[0]].sum():,.0f}")
    print(f"Average population per neighborhood: {neighborhoods_pop[pop_cols[0]].mean():,.0f}")
    print(f"Highest population: {neighborhoods_pop[pop_cols[0]].max():,.0f}")
    print(f"Lowest population: {neighborhoods_pop[pop_cols[0]].min():,.0f}")

## 6. Visualize Neighborhoods on Map

In [ ]:
# Calculate center of the map (Barcelona)
center_lat = neighborhoods_gdf.geometry.centroid.y.mean()
center_lon = neighborhoods_gdf.geometry.centroid.x.mean()

print(f"Map center: ({center_lat}, {center_lon})")

# Normalize data for color mapping if available
if len(income_cols) > 0:
    income_normalized = (income_df[income_cols[0]] - income_df[income_cols[0]].min()) / (income_df[income_cols[0]].max() - income_df[income_cols[0]].min())
else:
    income_normalized = None

In [ ]:
# Create interactive folium map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Add neighborhood polygons
for idx, row in neighborhoods_gdf.iterrows():
    # Get income value for color coding
    income_val = None
    if income_normalized is not None and idx < len(income_normalized):
        income_val = income_normalized.iloc[idx] if hasattr(income_normalized, 'iloc') else income_normalized[idx]
    
    # Color based on income normalized value (if available)
    if income_val is not None:
        color = plt.cm.RdYlGn(income_val)
        hex_color = '%02x%02x%02x' % (int(color[0]*255), int(color[1]*255), int(color[2]*255))
    else:
        hex_color = '0066cc'
    
    # Add polygon
    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x, color=hex_color: {
            'fillColor': '#' + color,
            'color': 'black',
            'weight': 2,
            'fillOpacity': 0.7
        }
    ).add_to(m)

# Save and display map
map_path = '../reports/figures/neighborhoods_map.html'
m.save(map_path)
print(f"Map saved to: {map_path}")
m

## 7. Summary Statistics & Correlation Analysis

In [ ]:
# Key Insights Summary
print("="*60)
print("EXPLORATORY DATA ANALYSIS SUMMARY - BARCELONA NEIGHBORHOODS")
print("="*60)

print("\n### GEOGRAPHIC DATA ###")
print(f"Total neighborhoods: {len(neighborhoods_gdf)}")
print(f"CRS: {neighborhoods_gdf.crs}")
print(f"Bounding box: {neighborhoods_gdf.total_bounds}")

print("\n### INCOME DATA ###")
if len(income_cols) > 0:
    for col in income_cols:
        print(f"\n{col}:")
        print(f"  Mean: {income_df[col].mean():.4f}")
        print(f"  Median: {income_df[col].median():.4f}")
        print(f"  Std Dev: {income_df[col].std():.4f}")
        print(f"  Min: {income_df[col].min():.4f}")
        print(f"  Max: {income_df[col].max():.4f}")

print("\n### POPULATION DATA ###")
if len(pop_cols) > 0:
    for col in pop_cols:
        print(f"\n{col}:")
        print(f"  Total: {neighborhoods_pop[col].sum():,.0f}")
        print(f"  Mean: {neighborhoods_pop[col].mean():,.0f}")
        print(f"  Median: {neighborhoods_pop[col].median():,.0f}")
        print(f"  Std Dev: {neighborhoods_pop[col].std():,.0f}")
        print(f"  Min: {neighborhoods_pop[col].min():,.0f}")
        print(f"  Max: {neighborhoods_pop[col].max():,.0f}")

In [ ]:
# Correlation analysis if we can merge data
print("\n\n### CORRELATION ANALYSIS ###")

# Try to merge income and population data for correlation
try:
    analysis_df = income_df.copy()
    if len(neighborhoods_pop) > 0 and len(pop_cols) > 0:
        # Ensure alignment
        if len(analysis_df) == len(neighborhoods_pop):
            for col in pop_cols:
                analysis_df[f'pop_{col}'] = neighborhoods_pop[col].values
        
        # Calculate correlations
        numeric_df = analysis_df.select_dtypes(include=[np.number])
        if len(numeric_df.columns) > 1:
            print("\nCorrelation Matrix:")
            corr_matrix = numeric_df.corr()
            print(corr_matrix)
            
            # Visualize correlation heatmap
            plt.figure(figsize=(10, 8))
            sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, square=True)
            plt.title('Correlation Matrix: Income & Population')
            plt.tight_layout()
            plt.show()
except Exception as e:
    print(f"Could not perform correlation analysis: {str(e)}")

print("\n" + "="*60)
print("EDA COMPLETE - Map saved to: ../reports/figures/neighborhoods_map.html")
print("="*60)